In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


def run_pca_lda_pipeline(
    train_path="./datasets/train",
    test_path="./datasets/test",
    img_size=(90, 90),
    k=40,
    show_mean_face=True,
    show_reconstruction=True,
    n_recon=5
):
    """
    Full PCA + LDA + 1-NN face recognition pipeline in one function.

    Steps:
    1. Load train/test images
    2. Flatten images
    3. Compute mean face
    4. Center data
    5. PCA using smaller covariance trick
    6. Project to PCA space
    7. Reconstruct sample images
    8. LDA on PCA features
    9. Project train/test to LDA space
    10. Classify test set using 1-NN
    11. Return results
    """

    def load_dataset(dataset_path):
        X = []
        y = []

        for person_name in sorted(os.listdir(dataset_path)):
            person_dir = os.path.join(dataset_path, person_name)

            if not os.path.isdir(person_dir):
                continue

            for file in os.listdir(person_dir):
                img_path = os.path.join(person_dir, file)

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                if img.shape != img_size:
                    img = cv2.resize(img, img_size)

                img = img.astype(np.float32) / 255.0
                X.append(img.flatten())
                y.append(person_name)

        return np.array(X), np.array(y)

    # =========================
    # 1. Load dataset
    # =========================
    X_train, y_train = load_dataset(train_path)
    X_test, y_test = load_dataset(test_path)

    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError("Train or test dataset is empty.")

    # =========================
    # 2. Mean face
    # =========================
    mean_face = np.mean(X_train, axis=0)

    if show_mean_face:
        plt.figure(figsize=(4, 4))
        plt.imshow(mean_face.reshape(img_size), cmap="gray")
        plt.title("Mean Face")
        plt.axis("off")
        plt.show()

    # =========================
    # 3. Center data
    # =========================
    X_train_centered = X_train - mean_face
    X_test_centered = X_test - mean_face

    # =========================
    # 4. PCA
    # =========================
    # smaller matrix trick: X X^T
    XtX = X_train_centered @ X_train_centered.T

    eigenvalues, eigenvectors_small = np.linalg.eigh(XtX)

    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors_small = eigenvectors_small[:, idx]

    # map back to original high-dimensional space
    eigenvectors_large = X_train_centered.T @ eigenvectors_small

    # avoid division by zero
    eigenvectors_large = eigenvectors_large / np.sqrt(eigenvalues + 1e-10)

    # normalize eigenvectors
    norms = np.linalg.norm(eigenvectors_large, axis=0, keepdims=True)
    norms[norms < 1e-12] = 1.0
    eigenvectors_large = eigenvectors_large / norms

    # keep top-k PCA components
    eigenvectors_reduced = eigenvectors_large[:, :k]

    # project to PCA space
    X_train_reduced = X_train_centered @ eigenvectors_reduced
    X_test_reduced = X_test_centered @ eigenvectors_reduced

    # =========================
    # 5. Reconstruction
    # =========================
    X_train_reconstructed = X_train_reduced @ eigenvectors_reduced.T + mean_face

    if show_reconstruction:
        n_show = min(n_recon, len(X_train))
        plt.figure(figsize=(2 * n_show, 4))
        for i in range(n_show):
            plt.subplot(2, n_show, i + 1)
            plt.imshow(X_train[i].reshape(img_size), cmap="gray")
            plt.title("Original")
            plt.axis("off")

            plt.subplot(2, n_show, n_show + i + 1)
            plt.imshow(X_train_reconstructed[i].reshape(img_size), cmap="gray")
            plt.title("Recon")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

    # =========================
    # 6. LDA
    # =========================
    N, d = X_train_reduced.shape
    classes = np.unique(y_train)
    overall_mean = np.mean(X_train_reduced, axis=0)

    Sw = np.zeros((d, d), dtype=np.float64)
    Sb = np.zeros((d, d), dtype=np.float64)

    for c in classes:
        X_c = X_train_reduced[y_train == c]
        mean_c = np.mean(X_c, axis=0)

        # within-class scatter
        Sw += (X_c - mean_c).T @ (X_c - mean_c)

        # between-class scatter
        n_c = X_c.shape[0]
        mean_diff = (mean_c - overall_mean).reshape(-1, 1)
        Sb += n_c * (mean_diff @ mean_diff.T)

    # regularization
    Sw += 1e-6 * np.eye(d)

    M = np.linalg.solve(Sw, Sb)
    eigvals, eigvecs = np.linalg.eig(M)

    eigvals = np.real(eigvals)
    eigvecs = np.real(eigvecs)

    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    lda_dim = len(classes) - 1
    W_lda = eigvecs[:, :lda_dim]

    # normalize LDA directions
    norms = np.linalg.norm(W_lda, axis=0, keepdims=True)
    norms[norms < 1e-12] = 1.0
    W_lda = W_lda / norms

    # project to LDA space
    X_train_lda = X_train_reduced @ W_lda
    X_test_lda = X_test_reduced @ W_lda

    # =========================
    # 7. 1-NN classifier
    # =========================
    y_pred = []

    for x in X_test_lda:
        dists = np.linalg.norm(X_train_lda - x, axis=1)
        idx = np.argmin(dists)
        y_pred.append(y_train[idx])

    y_pred = np.array(y_pred)
    accuracy = np.mean(y_pred == y_test)

    print("Accuracy:", accuracy)

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "mean_face": mean_face,
        "eigenvectors_pca": eigenvectors_reduced,
        "W_lda": W_lda,
        "X_train_pca": X_train_reduced,
        "X_test_pca": X_test_reduced,
        "X_train_lda": X_train_lda,
        "X_test_lda": X_test_lda,
        "y_pred": y_pred,
        "accuracy": accuracy,
    }

In [10]:
import cv2
import numpy as np


def recognize_face_realtime(
    mean_face,
    eigenvectors_pca,
    W_lda,
    y_train,
    X_train_features,
    classifier="knn",
    img_size=(90, 90),
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(60, 60),
    margin=10
):
    """
    Real-time face detection + recognition using webcam.

    Parameters
    ----------
    mean_face : ndarray, shape (D,)
        Mean face from training set.
    eigenvectors_pca : ndarray, shape (D, k)
        PCA projection matrix.
    W_lda : ndarray, shape (k, lda_dim)
        LDA projection matrix.
    y_train : ndarray, shape (N,)
        Training labels.
    X_train_features : ndarray, shape (N, lda_dim) or (N, k)
        Training features in the same space as classification is done.
        For PCA+LDA pipeline, this should be X_train_lda.
    classifier : str
        "knn" or "gaussian_shared"
    img_size : tuple
        Target face image size, e.g. (90, 90)
    scaleFactor, minNeighbors, minSize
        Haar cascade parameters.
    margin : int
        Extra pixels around detected face before cropping.
    """

    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

    if face_cascade.empty():
        raise RuntimeError("Failed to load Haar cascade.")

    # ----- optional Gaussian shared covariance precomputation -----
    if classifier == "gaussian_shared":
        classes = np.unique(y_train)
        class_means = {}

        for c in classes:
            class_means[c] = np.mean(X_train_features[y_train == c], axis=0)

        d = X_train_features.shape[1]
        Sigma = np.zeros((d, d), dtype=np.float64)

        for c in classes:
            X_c = X_train_features[y_train == c]
            mu_c = class_means[c]
            Sigma += (X_c - mu_c).T @ (X_c - mu_c)

        denom = max(len(X_train_features) - len(classes), 1)
        Sigma /= denom
        Sigma += 1e-6 * np.eye(d)
        Sigma_inv = np.linalg.inv(Sigma)

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Cannot open camera")
        return

    print("Press 'q' to quit")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to read frame from camera")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=scaleFactor,
            minNeighbors=minNeighbors,
            minSize=minSize
        )

        for (x, y, w, h) in faces:
            # safer crop with boundary clipping
            x1 = max(x - margin, 0)
            y1 = max(y - margin, 0)
            x2 = min(x + w + margin, frame.shape[1])
            y2 = min(y + h + margin, frame.shape[0])

            cropped_gray = gray[y1:y2, x1:x2]

            if cropped_gray.size == 0:
                continue

            # resize to match training image size
            resized_face = cv2.resize(cropped_gray, img_size)

            # normalize exactly like training
            face_vector = resized_face.astype(np.float32) / 255.0
            face_vector = face_vector.flatten()

            # PCA projection
            face_centered = face_vector - mean_face
            face_pca = face_centered @ eigenvectors_pca

            # LDA projection
            face_feature = face_pca @ W_lda

            # classification
            if classifier == "knn":
                dists = np.linalg.norm(X_train_features - face_feature, axis=1)
                pred_idx = np.argmin(dists)
                pred_label = y_train[pred_idx]

            elif classifier == "gaussian_shared":
                best_class = None
                best_score = -np.inf

                for c in classes:
                    mu_c = class_means[c]
                    diff = face_feature - mu_c
                    score = -0.5 * diff.T @ Sigma_inv @ diff

                    if score > best_score:
                        best_score = score
                        best_class = c

                pred_label = best_class

            else:
                raise ValueError("Unsupported classifier. Use 'knn' or 'gaussian_shared'.")

            # draw face box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # show predicted label
            cv2.putText(
                frame,
                str(pred_label),
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            # show cropped face
            cv2.imshow("Cropped Face", resized_face)

        cv2.imshow("Viola-Jones Face Detection + Recognition", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

In [18]:
results = run_pca_lda_pipeline(
    train_path="./datasets/train",
    test_path="./datasets/test",
    show_mean_face=False,
    show_reconstruction=False,
    img_size=(90, 90),
    k=20
)

Train shape: (65, 8100)
Test shape: (23, 8100)
Predicted labels: ['P_Fathiah' 'P_Fathiah' 'P_Fazil' 'P_Fazil' 'P_Joe' 'P_Jasper' 'P_Joe'
 'P_Syarah' 'P_Jasper' 'P_Syarah' 'P_Joe' 'P_Joe' 'P_Fathiah' 'P_Syarah'
 'P_Roy' 'P_Mikail' 'P_Roy' 'P_Jasper' 'P_Jasper' 'P_Fazil' 'P_Syarah'
 'P_Fazil' 'P_Roy']
True labels: ['P_Fathiah' 'P_Fathiah' 'P_Fazil' 'P_Fazil' 'P_Jasper' 'P_Jasper'
 'P_Jasper' 'P_Jasper' 'P_Jasper' 'P_Jasper' 'P_Joe' 'P_Joe' 'P_Joe'
 'P_Joe' 'P_Mikail' 'P_Mikail' 'P_Roy' 'P_Roy' 'P_Syarah' 'P_Syarah'
 'P_Syarah' 'P_Syarah' 'P_Syarah']
Accuracy: 0.4782608695652174


In [19]:
recognize_face_realtime(
    mean_face=results["mean_face"],
    eigenvectors_pca=results["eigenvectors_pca"],
    W_lda=results["W_lda"],
    y_train=results["y_train"],
    X_train_features=results["X_train_lda"],   # because classification is in LDA space
    classifier="gaussian_shared",
    img_size=(90, 90)
)

Press 'q' to quit


In [16]:
results['y_train']

array(['P_Fathiah', 'P_Fathiah', 'P_Fathiah', 'P_Fathiah', 'P_Fathiah',
       'P_Fathiah', 'P_Fazil', 'P_Fazil', 'P_Fazil', 'P_Fazil', 'P_Fazil',
       'P_Fazil', 'P_Fazil', 'P_Fazil', 'P_Fazil', 'P_Fazil', 'P_Jasper',
       'P_Jasper', 'P_Jasper', 'P_Jasper', 'P_Jasper', 'P_Jasper',
       'P_Jasper', 'P_Jasper', 'P_Jasper', 'P_Jasper', 'P_Joe', 'P_Joe',
       'P_Joe', 'P_Joe', 'P_Joe', 'P_Joe', 'P_Joe', 'P_Joe', 'P_Joe',
       'P_Joe', 'P_Mikail', 'P_Mikail', 'P_Mikail', 'P_Mikail',
       'P_Mikail', 'P_Mikail', 'P_Mikail', 'P_Mikail', 'P_Mikail',
       'P_Mikail', 'P_Roy', 'P_Roy', 'P_Roy', 'P_Roy', 'P_Roy', 'P_Roy',
       'P_Roy', 'P_Roy', 'P_Roy', 'P_Syarah', 'P_Syarah', 'P_Syarah',
       'P_Syarah', 'P_Syarah', 'P_Syarah', 'P_Syarah', 'P_Syarah',
       'P_Syarah', 'P_Syarah'], dtype='<U9')